In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading"
)

IMAGE_DATA_DIR = PROJECT_DIR / "ml/grading/image/data/processed"
IMAGE_DIR = IMAGE_DATA_DIR / "images"

MANIFEST_PATH = IMAGE_DATA_DIR / "clean_image_manifest.csv"
SPLIT_PATH = IMAGE_DATA_DIR / "splits/multimodal_shared_split.csv"

manifest = pd.read_csv(MANIFEST_PATH)
shared_split = pd.read_csv(SPLIT_PATH)

print("Manifest shape:", manifest.shape)
print("Shared split shape:", shared_split.shape)
print("\nSplit distribution:")
print(pd.crosstab(shared_split["Split"], shared_split["Tissue_Grade_Clean"]))

assert shared_split["Normalized_ID"].is_unique
assert set(shared_split["Split"]) == {"train", "validation", "test"}
assert set(shared_split["Tissue_Grade_Clean"].dropna()) == {"A", "A+", "B"}

print("\nShared split loaded successfully.")

Manifest shape: (1024, 9)
Shared split shape: (1009, 3)

Split distribution:
Tissue_Grade_Clean    A   A+    B
Split                            
test                 47   80   25
train               220  371  115
validation           47   79   25

Shared split loaded successfully.


In [2]:
# Keep only the labelled records included in the shared split
image_df = shared_split.merge(
    manifest[["Normalized_ID", "Selected_Image"]],
    on="Normalized_ID",
    how="left",
    validate="one_to_one"
)

# Validate image paths
image_df["Image_Path"] = image_df["Selected_Image"].apply(
    lambda p: Path(p) if pd.notna(p) else None
)

missing_paths = image_df["Image_Path"].isna().sum()
missing_files = sum(
    not p.exists()
    for p in image_df["Image_Path"]
    if p is not None
)

print("Dataset shape:", image_df.shape)
print("\nSplit counts:")
print(image_df["Split"].value_counts())
print("\nMissing image paths:", missing_paths)
print("Missing image files:", missing_files)
print("\nClass distribution:")
print(pd.crosstab(image_df["Split"], image_df["Tissue_Grade_Clean"]))

assert missing_paths == 0, "Some records have no image path."
assert missing_files == 0, "Some image files are missing."

print("\nImage-only dataset is ready.")

Dataset shape: (1009, 5)

Split counts:
Split
train         706
test          152
validation    151
Name: count, dtype: int64

Missing image paths: 0
Missing image files: 0

Class distribution:
Tissue_Grade_Clean    A   A+    B
Split                            
test                 47   80   25
train               220  371  115
validation           47   79   25

Image-only dataset is ready.


In [3]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

CLASS_NAMES = ["A", "A+", "B"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

IMAGE_SIZE = 224
BATCH_SIZE = 16


class CorneaImageDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]

        with Image.open(row["Image_Path"]) as img:
            img = img.convert("L")
            img.thumbnail(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS
            )

            canvas = Image.new(
                "L", (IMAGE_SIZE, IMAGE_SIZE), color=0
            )
            left = (IMAGE_SIZE - img.width) // 2
            top = (IMAGE_SIZE - img.height) // 2
            canvas.paste(img, (left, top))

            image_array = (
                np.asarray(canvas, dtype=np.float32) / 255.0
            )

        image_tensor = torch.from_numpy(
            image_array[np.newaxis, :, :].copy()
        )
        label = CLASS_TO_IDX[row["Tissue_Grade_Clean"]]

        return image_tensor, label


class CorneaCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Create datasets using the shared split
train_rows = image_df[image_df["Split"] == "train"]
val_rows = image_df[image_df["Split"] == "validation"]
test_rows = image_df[image_df["Split"] == "test"]

train_dataset = CorneaImageDataset(train_rows)
val_dataset = CorneaImageDataset(val_rows)
test_dataset = CorneaImageDataset(test_rows)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

sample_image, sample_label = train_dataset[0]
print("Sample image shape:", sample_image.shape)
print("Sample label:", CLASS_NAMES[sample_label])

model = CorneaCNN().to(device)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Device: cuda
Train images: 706
Validation images: 151
Test images: 152
Sample image shape: torch.Size([1, 224, 224])
Sample label: A+
Trainable parameters: 23715


In [4]:
from sklearn.metrics import f1_score
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=torch.Generator().manual_seed(SEED)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

MAX_EPOCHS = 30
PATIENCE = 5

print("Training configuration ready.")
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Optimizer: Adam (lr=0.001, weight_decay=0.0001)")
print("Loss: CrossEntropyLoss")

Training configuration ready.
Batch size: 16
Maximum epochs: 30
Early-stopping patience: 5
Optimizer: Adam (lr=0.001, weight_decay=0.0001)
Loss: CrossEntropyLoss


In [5]:
import copy
import time
import pandas as pd
from sklearn.metrics import f1_score

best_val_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0
history = []

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    # Training
    model.train()
    train_loss_sum = 0.0
    train_true, train_pred = [], []

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        predictions = logits.argmax(dim=1)

        train_true.extend(labels.cpu().tolist())
        train_pred.extend(predictions.cpu().tolist())

    train_loss = train_loss_sum / len(train_dataset)
    train_f1 = f1_score(
        train_true, train_pred,
        average="macro",
        zero_division=0
    )

    # Validation
    model.eval()
    val_loss_sum = 0.0
    val_true, val_pred = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            predictions = logits.argmax(dim=1)

            val_true.extend(labels.cpu().tolist())
            val_pred.extend(predictions.cpu().tolist())

    val_loss = val_loss_sum / len(val_dataset)
    val_f1 = f1_score(
        val_true, val_pred,
        average="macro",
        zero_division=0
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_macro_f1": val_f1
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    # Save best checkpoint using validation macro F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

# Restore best validation checkpoint
model.load_state_dict(best_state)

history_df = pd.DataFrame(history)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation macro F1: {best_val_f1:.4f}")
print(f"Training time: {(time.time() - start_time) / 60:.2f} minutes")

Epoch 01/30 | Train Loss: 1.0328 | Train F1: 0.2831 | Val Loss: 1.0642 | Val Macro F1: 0.2290
Epoch 02/30 | Train Loss: 1.0209 | Train F1: 0.2474 | Val Loss: 0.9794 | Val Macro F1: 0.2290
Epoch 03/30 | Train Loss: 1.0056 | Train F1: 0.2515 | Val Loss: 0.9806 | Val Macro F1: 0.2290
Epoch 04/30 | Train Loss: 0.9919 | Train F1: 0.2591 | Val Loss: 1.0147 | Val Macro F1: 0.2290
Epoch 05/30 | Train Loss: 0.9951 | Train F1: 0.2505 | Val Loss: 1.1086 | Val Macro F1: 0.2920
Epoch 06/30 | Train Loss: 0.9929 | Train F1: 0.2409 | Val Loss: 0.9829 | Val Macro F1: 0.2290
Epoch 07/30 | Train Loss: 0.9842 | Train F1: 0.2649 | Val Loss: 0.9472 | Val Macro F1: 0.2556
Epoch 08/30 | Train Loss: 0.9444 | Train F1: 0.3124 | Val Loss: 1.0850 | Val Macro F1: 0.2290
Epoch 09/30 | Train Loss: 0.9746 | Train F1: 0.2910 | Val Loss: 0.9569 | Val Macro F1: 0.4170
Epoch 10/30 | Train Loss: 0.9363 | Train F1: 0.2891 | Val Loss: 1.4204 | Val Macro F1: 0.2290
Epoch 11/30 | Train Loss: 0.9499 | Train F1: 0.3452 | Val Lo

In [6]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)

        logits = model(images)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())

print("Test samples:", len(y_true))
print(f"Accuracy:          {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1:          {f1_score(y_true, y_pred, average='macro'):.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

Test samples: 152
Accuracy:          0.5461
Balanced accuracy: 0.3960
Macro F1:          0.3706

Classification report:
              precision    recall  f1-score   support

           A     0.3590    0.2979    0.3256        47
          A+     0.6126    0.8500    0.7120        80
           B     0.5000    0.0400    0.0741        25

    accuracy                         0.5461       152
   macro avg     0.4905    0.3960    0.3706       152
weighted avg     0.5157    0.5461    0.4876       152

Confusion matrix (rows = actual, columns = predicted):
[[14 32  1]
 [12 68  0]
 [13 11  1]]
